In [1]:
from tqdm.notebook import tqdm

In [2]:
from transformers import AutoTokenizer

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\transformers\utils\hub.py:123: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
model = "Helsinki-NLP/opus-mt-zh-en"
tokenizer = AutoTokenizer.from_pretrained(model)

In [4]:
from datasets import load_dataset

In [5]:
dataset_name = "wmt19"

In [6]:
subset = "zh-en"

In [7]:
wm = load_dataset(dataset_name, subset)

Extracting data files:   0%|          | 0/10 [00:00<?, ?it/s]

Extracting data files: 0it [00:00, ?it/s]

Generating train split:   0%|          | 0/25984574 [00:00<?, ? examples/s]

Skipping line 9877 in TSV (G:\AppData\huggingface\downloads\extracted\68b36ff4746646bf4a77ba6069873ca3337a5cca7d3d7eeac6e54706d520374a) with 3 != 2 columns.
Skipping line 9879 in TSV (G:\AppData\huggingface\downloads\extracted\68b36ff4746646bf4a77ba6069873ca3337a5cca7d3d7eeac6e54706d520374a) with 3 != 2 columns.
Skipping line 9885 in TSV (G:\AppData\huggingface\downloads\extracted\68b36ff4746646bf4a77ba6069873ca3337a5cca7d3d7eeac6e54706d520374a) with 3 != 2 columns.
Skipping line 9895 in TSV (G:\AppData\huggingface\downloads\extracted\68b36ff4746646bf4a77ba6069873ca3337a5cca7d3d7eeac6e54706d520374a) with 3 != 2 columns.
Skipping line 9897 in TSV (G:\AppData\huggingface\downloads\extracted\68b36ff4746646bf4a77ba6069873ca3337a5cca7d3d7eeac6e54706d520374a) with 3 != 2 columns.
Skipping line 9899 in TSV (G:\AppData\huggingface\downloads\extracted\68b36ff4746646bf4a77ba6069873ca3337a5cca7d3d7eeac6e54706d520374a) with 3 != 2 columns.
Skipping line 9901 in TSV (G:\AppData\huggingface\download

Generating validation split:   0%|          | 0/3981 [00:00<?, ? examples/s]

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\datasets\table.py:1421: FutureWarning: promote has been superseded by mode='default'.
  table = cls._concat_blocks(blocks, axis=0)


In [9]:
wm

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 25984574
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 3981
    })
})

In [10]:
wm_train = wm["train"]

In [11]:
sample = wm_train[0]

In [12]:
sample

{'translation': {'en': '1929 or 1989?', 'zh': '1929年还是1989年?'}}

In [13]:
tokenizer.encode(sample["translation"]["en"])

[1013, 945, 7, 672, 8134, 23, 0]

In [14]:
sample # {'translation': {'en': '1929 or 1989?', 'zh': '1929年还是1989年?'}}

{'translation': {'en': '1929 or 1989?', 'zh': '1929年还是1989年?'}}

In [15]:
ens = []
zhs = []

In [16]:
max_len = 99

In [17]:
# tokenizer doesn't have a bos token, so add it manually
bos = "<s>"
tokenizer.add_special_tokens({"bos_token": bos})

1

In [18]:
for sample in tqdm(wm_train):
    en = sample["translation"]["en"]
    zh = sample["translation"]["zh"]
    en = tokenizer.encode(en)
    zh = tokenizer.encode(zh)
    if len(en) > max_len or len(zh) > max_len:
        continue
    else:
        # add bos token
        en = [tokenizer.bos_token_id] + en
        zh = [tokenizer.bos_token_id] + zh
        ens.append(en)
        zhs.append(zh)

  0%|          | 0/25984574 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [18]:
import torch

In [19]:
ens = torch.nested.nested_tensor(ens)
zhs = torch.nested.nested_tensor(zhs)

C:\Users\John\AppData\Local\Temp\ipykernel_14188\3170439811.py:1: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  ens = torch.nested.nested_tensor(ens)


In [20]:
data_dir = "data"
file_name = "processed3.pt"

In [21]:
pad_idx = tokenizer.pad_token_id

In [22]:
ens = torch.nested.to_padded_tensor(ens, pad_idx)

In [23]:
zhs = torch.nested.to_padded_tensor(zhs, pad_idx)

In [24]:
ens.shape

torch.Size([220722, 100])

In [25]:
torch.save((ens, zhs), f"{data_dir}/{file_name}")